# Raw Raman conversion and APOLLO-inspired preprocessing

This notebook merges the individual image `.npy` files in `data/RawData`, audits them against `preprocessed.npy`, verifies the APOLLO spectral transform on sampled pixels, and compares spectrum-wise preprocessing choices with patient-level GPR. Implementation is in [`raw_preprocessing.py`](../src/glioma_raman_gp_regression/raw_preprocessing.py); methods and results are in [`05_raw_apollo_preprocessing.md`](05_raw_apollo_preprocessing.md).

The independent observation and cross-validation split unit is the patient. All 46 patients remain in the analysis.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'data' / 'preprocessed.npy').exists():
    repo_root = repo_root.parent
data_dir = repo_root / 'data'
raw_root = data_dir / 'RawData'
processed_path = data_dir / 'preprocessed.npy'
merged_path = data_dir / 'raw_merged.npy'
sys.path.insert(0, str(repo_root / 'src'))
from glioma_raman_gp_regression.raw_preprocessing import (
    merge_raw_dataset, audit_merged_dataset, verify_apollo_preprocessing,
    build_preprocessing_features, evaluate_preprocessing_gpr,
)

print('Repository root:', repo_root)
print('Raw source:', raw_root)
print('Merged file:', merged_path)

## Merge raw arrays and verify sample identity

The merge follows current `preprocessed.npy` entry order. Metadata comes from both sources: sample name from the filename stem, class ID from its `LGm-N` directory, and patient ID from the `HF-<digits>` prefix. The routine also checks image dimensions and verifies each merged array against its source with SHA-256.

In [ ]:
if not merged_path.exists():
    merge_manifest = merge_raw_dataset(raw_root, processed_path, merged_path)
    print(f'Created merged dataset with {len(merge_manifest)} entries.')
else:
    print('Merged file exists; checking it against all raw sources.')

conversion_audit = audit_merged_dataset(raw_root, processed_path, merged_path)
conversion_audit.to_csv(data_dir / 'raw_merged_audit.csv', index=False)
print('Entries verified:', len(conversion_audit))
print('Unique patients:', conversion_audit.patient_id.nunique())
print('Class-folder counts:', conversion_audit.groupby('class_id').size().to_dict())
print('Exact source-array content:', bool(conversion_audit.array_content_exact.all()))
print('Raw/processed spatial shapes match:', bool(all(tuple(r.raw_shape[:2]) == tuple(r.processed_shape[:2]) for r in conversion_audit.itertuples())))

## Reproduce the APOLLO spectrum transform

Lita et al. describe removing the low-wavenumber and silent regions, applying airPLS (lambda 4, p-order 1) to each retained segment, clipping negative residuals, concatenating the segments, and L2-normalizing each spectrum. The saved raw arrays do not include a wavenumber calibration vector; the channel slices below produce the reported 1,167 channels and are checked against the current processed arrays. This cell samples five pixels from each image rather than reprocessing every pixel.

In [ ]:
apollo_check = verify_apollo_preprocessing(merged_path, processed_path, samples_per_entry=5)
apollo_check.to_csv(data_dir / 'raw_apollo_pixel_check.csv', index=False)
print('Entries checked:', len(apollo_check))
print('Sampled pixels:', int(apollo_check.sampled_pixels.sum()))
print('Worst absolute difference:', f"{apollo_check.max_absolute_error.max():.3e}")
print('Lowest per-entry median cosine similarity:', f"{apollo_check.median_cosine_similarity.min():.12f}")

## Compare preprocessing choices with GPR

Every transform is applied per spectrum before pooling spectra to a patient mean. The tested alternatives are the current APOLLO output, raw crop/clip/L2 without the software airPLS step, pixel-wise SNV, Savitzky–Golay smoothing, and the first derivative. PCA 8 and ARD RBF GPR are held fixed to isolate preprocessing effects; PCA and GPR are refit within each patient training fold. Seed 42 reproduces notebook 03's fixed-fold baseline.

In [ ]:
features = build_preprocessing_features(merged_path, processed_path)
preprocessing_summary, preprocessing_by_seed = evaluate_preprocessing_gpr(features)
preprocessing_summary.to_csv(data_dir / 'raw_preprocessing_gpr_summary.csv', index=False)
preprocessing_by_seed.to_csv(data_dir / 'raw_preprocessing_gpr_by_seed.csv', index=False)
print('Pooled patient-level OOF metrics; mean and SD across seeds:')
print(preprocessing_summary.to_string(index=False, float_format=lambda value: f'{value:.3f}'))
print('\nPer-seed OOF metrics:')
print(preprocessing_by_seed.to_string(index=False, float_format=lambda value: f'{value:.3f}'))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(
    preprocessing_summary['rmse_mean'], preprocessing_summary['roc_auc_mean'],
    xerr=preprocessing_summary['rmse_std'], yerr=preprocessing_summary['roc_auc_std'],
    fmt='o', capsize=3, alpha=0.85,
)
for _, row in preprocessing_summary.iterrows():
    ax.annotate(row['preprocessing'], (row['rmse_mean'], row['roc_auc_mean']), xytext=(5, 3), textcoords='offset points', fontsize=8)
ax.set_xlabel('Patient-level RMSE (lower is better)')
ax.set_ylabel('Pooled patient-level ROC AUC')
ax.set_title('GPR preprocessing sensitivity across three split seeds')
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Interpretation

The derivative is the strongest candidate in this exploratory comparison, with a small improvement in mean AUC/RMSE and no mean accuracy gain. The raw no-airPLS and SNV variants show weaker predictive-density diagnostics on some splits. Regression-derived threshold mass is not calibrated binary probability. The full tables, APOLLO notes, and limitations are in the companion report.